In [0]:
#%skip
%pip install Faker

In [0]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
import random
from faker import Faker

try:
    #os.remove('/Workspace/Users/nina.merkt@abat.de/test/data/chat_transcripts.csv')
    os.remove('/Workspace/Users/nina.merkt@abat.de/test/data/churn_labels.csv')
    os.remove('/Workspace/Users/nina.merkt@abat.de/test/data/connection_quality_logs.csv')
    os.remove('/Workspace/Users/nina.merkt@abat.de/test/data/customer_profile.csv')
    #os.remove('/Workspace/Users/nina.merkt@abat.de/test/data/support_tickets.csv')
    os.remove ('/Workspace/Users/nina.merkt@abat.de/test/data/agent_profile.csv')
except FileNotFoundError:
    pass

# Seed für Reproduzierbarkeit
np.random.seed(42)
random.seed(42)

# Instanz für FakeXYZ
fake = Faker('de_DE')

# Generate Customer Profile

In [0]:
n_customers = 1000

locations = []
for _ in range(n_customers):
    locations.append(fake.address())
addresses = pd.DataFrame(locations)

customer_profile = pd.DataFrame({
    'customer_id': [f'CUST_{i:05d}' for i in range(1, n_customers + 1)],
    'signup_date': pd.date_range(end='2026-03-01', periods=n_customers, freq='2D'),
    'plan_tier': np.random.choice(['Basic_50Mbps', 'Standard_200Mbps', 'Premium_1Gbps'], 
                                  n_customers, p=[0.35, 0.45, 0.2]),
    'address': addresses.values.flatten(),
    'contract_type': np.random.choice(['monthly', 'annual', '2-year'], 
                                     n_customers, p=[0.5, 0.3, 0.2]),
    'autopay_enabled': np.random.choice([True, False], n_customers, p=[0.7, 0.3])
})

customer_profile['account_age_months'] = (
    (pd.Timestamp('2026-03-01') - customer_profile['signup_date']).dt.days / 30
).round(1)

plan_prices = {'Basic_50Mbps': 49.99, 'Standard_200Mbps': 79.99, 'Premium_1Gbps': 119.99}
customer_profile['monthly_bill'] = customer_profile['plan_tier'].map(plan_prices)
discount_customers = np.random.choice([True, False], n_customers, p=[0.2, 0.8])
customer_profile.loc[discount_customers, 'monthly_bill'] *= 0.9


speed_tiers = {'Basic_50Mbps': 50, 'Standard_200Mbps': 200, 'Premium_1Gbps': 1000}
customer_profile['speed_tier_mbps'] = customer_profile['plan_tier'].map(speed_tiers)
customer_profile['data_usage_gb_last_month'] = np.random.exponential(300, n_customers).round(1)

print("Customer Profile created")

# Generate Chrun Labels

In [0]:
churn_rate = 0.15
n_churned = int(n_customers * churn_rate)
churned_customers = np.random.choice(customer_profile['customer_id'], n_churned, replace=False)

churn_labels = pd.DataFrame({
    'customer_id': customer_profile['customer_id'],
    'churned': customer_profile['customer_id'].isin(churned_customers).astype(int)
})

churn_labels['churn_date'] = None
churn_labels.loc[churn_labels['churned'] == 1, 'churn_date'] = [
    datetime(2026, 3, 1) - timedelta(days=random.randint(1, 90)) 
    for _ in range(n_churned)
]

churn_reasons = ['competitor_price', 'poor_service', 'technical_issues', 'relocation', 'price_increase']
churn_labels.loc[churn_labels['churned'] == 1, 'churn_reason'] = np.random.choice(
    churn_reasons, n_churned, p=[0.35, 0.25, 0.20, 0.15, 0.05]
)

print("Churn Labels erstellt")

# Generate Connection Logs 

In [0]:
connection_logs = []
log_id = 1

# Dictionary um technische Probleme pro Kunde zu tracken
customer_technical_issues = {}

for customer_id in customer_profile['customer_id']:
    customer_data = customer_profile[customer_profile['customer_id'] == customer_id].iloc[0]
    is_churned = churn_labels[churn_labels['customer_id'] == customer_id]['churned'].values[0]
    
    # Initialisiere Issue-Liste für diesen Kunden
    customer_technical_issues[customer_id] = []
    
    # Mehr Logs für Kunden mit Problemen
    n_logs = random.randint(20, 50)
    
    for i in range(n_logs):
        timestamp = datetime(2026, 3, 1) - timedelta(
            days=random.randint(1, 90),
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59),
            seconds=random.randint(0, 59)
        )
        
        # Baseline Qualität (gut)
        speed_factor = random.uniform(0.85, 1.0)
        packet_loss = random.uniform(0, 2)
        latency = random.uniform(10, 40)
        downtime = 0
        drops = 0
        
        # Zufällige technische Probleme mit bestimmter Wahrscheinlichkeit
        problem_type = None
        
        # 20% Chance für technische Probleme (höher bei abgewanderten Kunden)
        problem_chance = 0.35 if is_churned else 0.15
        
        if random.random() < problem_chance:
            problem_type = np.random.choice([
                'slow_speed',
                'connection_drops',
                'high_latency',
                'outage',
                'packet_loss'
            ], p=[0.3, 0.15, 0.25, 0.1, 0.20])
            
            # Simuliere verschiedene Problemtypen
            if problem_type == 'slow_speed':
                speed_factor = random.uniform(0.2, 0.5)  # Nur 20-50% des Speeds
                latency = random.uniform(40, 100)
                
            elif problem_type == 'connection_drops':
                drops = random.randint(5, 25)
                packet_loss = random.uniform(5, 20)
                
            elif problem_type == 'high_latency':
                latency = random.uniform(100, 300)
                packet_loss = random.uniform(3, 10)
                
            elif problem_type == 'outage':
                downtime = random.randint(30, 180)
                speed_factor = 0
                drops = random.randint(10, 50)
                
            elif problem_type == 'packet_loss':
                packet_loss = random.uniform(10, 30)
                latency = random.uniform(60, 150)
            
            # Speichere dieses Problem-Event
            customer_technical_issues[customer_id].append({
                'timestamp': timestamp,
                'problem_type': problem_type,
                'log_id': f'LOG_{log_id:07d}',
                'severity': 'high' if downtime > 60 or speed_factor < 0.3 else 'medium'
            })
            

        connection_logs.append({
            'timestamp': timestamp,
            'issue_detected': 'error: ' + problem_type if problem_type else 'none',
            'customer_id': customer_id,
            'speed_measured_mbps': round(customer_data['speed_tier_mbps'] * speed_factor, 1),
            'packet_loss_percent': round(packet_loss, 2),
            'latency_ms': round(latency, 1),
            'downtime_minutes': downtime,
            'connection_drops_count': drops
        })
        
        log_id += 1

connection_quality_logs = pd.DataFrame(connection_logs)

print(f"Connection Logs erstellt: {len(connection_quality_logs)} Logs")
print(f"Davon mit Problemen: {len(connection_quality_logs[connection_quality_logs['issue_detected'] != 'none'])}")


# Generate Support Tickets

In [0]:
%skip
ticket_templates = {
    'slow_speed': [
        "Internet is extremely slow, only getting {actual_speed} Mbps instead of {expected_speed} Mbps",
        "Speed test shows terrible results - {actual_speed} Mbps when I'm paying for {expected_speed} Mbps",
        "Pages are loading very slowly, speed is way below what was promised",
        "Streaming keeps buffering, internet speed is insufficient"
    ],
    'connection_drops': [
        "Internet keeps disconnecting every few minutes",
        "Connection drops constantly, can't stay online",
        "WiFi keeps cutting out throughout the day",
        "Losing connection repeatedly, very frustrating"
    ],
    'high_latency': [
        "Terrible lag during video calls, latency is {latency}ms",
        "High ping, gaming is impossible with this latency",
        "Response time is awful, everything is delayed",
        "Video conferences are choppy, high latency issues"
    ],
    'outage': [
        "Complete outage for {downtime} minutes, no internet at all",
        "Internet has been down for over an hour",
        "Total service interruption in my area",
        "No connection whatsoever, lights on router are red"
    ],
    'packet_loss': [
        "Experiencing severe packet loss ({packet_loss}%), connection is unstable",
        "Data is getting lost, packet loss is very high",
        "Connection quality is terrible, lots of packet loss",
        "Unstable connection with frequent data loss"
    ],
    'billing': [
        "Charged ${amount} but my plan should be ${plan_price}",
        "Why did my bill increase without notice?",
        "Double charged this month",
        "Need explanation for additional fees on bill"
    ],
    'service_change': [
        "Want to upgrade to faster plan",
        "Need to downgrade to save money",
        "How do I cancel my service?",
        "Moving to new address, how to transfer service?"
    ],
    'sales': [
        "Competitor is offering better price, can you match?",
        "What promotions do you currently have?",
        "Interested in bundling services"
    ]
}

tickets = []
ticket_id = 1

for customer_id in customer_profile['customer_id']:
    customer_data = customer_profile[customer_profile['customer_id'] == customer_id].iloc[0]
    is_churned = churn_labels[churn_labels['customer_id'] == customer_id]['churned'].values[0]
    
    # Hole technische Probleme für diesen Kunden
    tech_issues = customer_technical_issues[customer_id]
    
    # Generiere Tickets basierend auf tatsächlichen Problemen
    # 70% der technischen Probleme führen zu einem Support Ticket
    tickets_from_issues = [issue for issue in tech_issues if random.random() < 0.7]
    
    # Füge noch zufällige andere Tickets hinzu (billing, service_change, etc.)
    if is_churned:
        n_other_tickets = random.randint(2, 5)
    else:
        n_other_tickets = random.randint(0, 3)
    
    # 1. TICKETS BASIEREND AUF TECHNISCHEN PROBLEMEN
    for issue in tickets_from_issues:
        problem_type = issue['problem_type']
        
        # Ticket wird 0-24 Stunden nach dem Problem erstellt
        ticket_timestamp = issue['timestamp'] + timedelta(hours=random.randint(0, 12))
        
        template = random.choice(ticket_templates[problem_type])
        
        # Hole die tatsächlichen Werte aus dem Connection Log
        log_data = connection_quality_logs[
            connection_quality_logs['timestamp'] == issue['log_id']
        ].iloc[0]
        
        description = template.format(
            actual_speed=int(log_data['speed_measured_mbps']),
            expected_speed=customer_data['speed_tier_mbps'],
            latency=int(log_data['latency_ms']),
            downtime=log_data['downtime_minutes'],
            packet_loss=round(log_data['packet_loss_percent'], 1)
        )
        
        # Technische Tickets dauern länger zu lösen
        if issue['severity'] == 'high':
            solved_hours = random.randint(12, 120)
            priority = random.choice(['high', 'urgent'])
        else:
            solved_hours = random.randint(1, 48)
            priority = random.choice(['medium', 'high'])
        
        timestamp_closed = ticket_timestamp + timedelta(hours=solved_hours)
        
        tickets.append({
            'ticket_id': f'TKT_{ticket_id:06d}',
            'customer_id': customer_id,
            'timestamp_created': ticket_timestamp,
            'timestamp_closed': timestamp_closed,
            'subject': f"Technical Issue - {problem_type.replace('_', ' ').title()}",
            'description': description,
            'category': 'technical',
            'priority': priority,
            'channel': random.choice(['email', 'chat', 'phone', 'web_form']),
            'status': 'closed'
        })
        ticket_id += 1
    
    # 2. ANDERE TICKETS (billing, service_change, sales)
    for _ in range(n_other_tickets):
        category = random.choices(
            ['billing', 'service_change', 'sales'],
            weights=[0.5, 0.35, 0.15]
        )[0]
        
        # Abgewanderte Kunden haben mehr "service_change" Tickets
        if is_churned and random.random() > 0.4:
            category = 'service_change'
        
        template = random.choice(ticket_templates[category])
        
        description = template.format(
            amount=round(customer_data['monthly_bill'] * random.uniform(1.1, 1.5), 2),
            plan_price=customer_data['monthly_bill']
        )
        
        timestamp_created = datetime(2026, 3, 1) - timedelta(days=random.randint(1, 180))
        
        if category == 'billing':
            solved_hours = random.randint(1, 48)
            priority = random.choice(['low', 'medium', 'high'])
        else:
            solved_hours = random.randint(1, 24)
            priority = random.choice(['low', 'medium'])
        
        timestamp_closed = timestamp_created + timedelta(hours=solved_hours)
        
        tickets.append({
            'ticket_id': f'TKT_{ticket_id:06d}',
            'customer_id': customer_id,
            'timestamp_created': timestamp_created,
            'timestamp_closed': timestamp_closed,
            'subject': category.replace('_', ' ').title(),
            'description': description,
            'category': category,
            'priority': priority,
            'channel': random.choice(['email', 'chat', 'phone', 'web_form']),
            'status': 'closed'
        })
        
        ticket_id += 1

support_tickets = pd.DataFrame(tickets)

print(f"Support Tickets erstellt: {len(support_tickets)} Tickets")
print(f"Davon technische Tickets: {len(support_tickets[support_tickets['category'] == 'technical'])}")
print(f"Mit Log-Verknüpfung: {support_tickets['related_log_id'].notna().sum()}")

# Generate Chat Transcripts

In [0]:
%skip
chat_messages_pool = {
    "customer_frustrated_technical": [
        "This is the third time this month my internet has gone out!",
        "I can't work from home with this unstable connection",
        "Your service is terrible, the speed is way below what you promised",
        "I'm paying for gigabit but only getting {speed} Mbps!",
        "The connection keeps dropping, this is unacceptable",
    ],
    "customer_neutral_technical": [
        "Hi, I'm having issues with my internet connection",
        "My router stopped working this morning",
        "Can you check if there's an outage in my area?",
        "The internet speed seems slower than usual",
        "I'm experiencing frequent disconnections",
    ],
    "customer_frustrated_billing": [
        "Why was I overcharged this month?",
        "I'm so frustrated, I can't work from home like this",
        "This billing error needs to be fixed immediately",
        "I've been a customer for years and this is how you treat me?",
    ],
    "customer_neutral": [
        "I'd like to know more about upgrading my plan",
        "What's the status of my support ticket?",
        "Can you help me understand my bill?",
    ],
    "customer_polite": [
        "Hello, I hope you can help me with a small issue",
        "Thank you for your time, I appreciate the help",
        "I understand, is there anything we can do about this?",
        "That makes sense, thanks for explaining",
    ],
    "agent_response_technical": [
        "I'm sorry to hear about your connection issues. Let me check your connection logs.",
        "I can see there was degraded service in your area. Let me investigate further.",
        "Your connection quality logs show {issue_type}. Let me escalate this.",
        "I see the problem in our system. We'll get a technician to look at this.",
        "Have you tried rebooting your router? This often resolves connection issues.",
        "I can see multiple connection drops on {date}. This isn't normal.",
    ],
    "agent_response_general": [
        "I understand your frustration. We'll get this resolved.",
        "Let me check your account details.",
        "I can offer you a discount on your next bill as compensation.",
        "Let me escalate this to our technical team.",
        "Is there anything else I can help with today?",
    ],
}

chat_transcripts = []
session_id = 1

customers_with_chats = np.random.choice(
    customer_profile["customer_id"], int(n_customers * 0.3), replace=False
)

for customer_id in customers_with_chats:
    is_churned = churn_labels[churn_labels["customer_id"] == customer_id][
        "churned"
    ].values[0]
    tech_issues = customer_technical_issues[customer_id]

    # Manche Chats sind wegen technischen Problemen
    n_technical_chats = min(len(tech_issues), random.randint(1, 3))
    n_other_chats = random.randint(0, 2) if not is_churned else random.randint(1, 3)

    # Chats basierend auf technischen Problemen
    for issue in random.sample(tech_issues, min(n_technical_chats, len(tech_issues))):
        timestamp_start = issue["timestamp"] + timedelta(hours=random.randint(0, 12))

        messages = []

        # Kunde beschreibt das Problem
        problem_desc = random.choice(
            chat_messages_pool[
                (
                    "customer_frustrated_technical"
                    if is_churned
                    else "customer_neutral_technical"
                )
            ]
        )

        # Füge tatsächliche Speed-Werte ein wenn im Template
        log_data = connection_quality_logs[
            connection_quality_logs["timestamp"] == issue["log_id"]
        ].iloc[0]

        if "{speed}" in problem_desc:
            problem_desc = problem_desc.format(
                speed=int(log_data["speed_measured_mbps"])
            )

        messages.append(
            {
                "speaker": "customer",
                "timestamp": timestamp_start.isoformat(),
                "message": problem_desc,
            }
        )

        # Agent antwortet mit Bezug zu Logs
        agent_msg = random.choice(chat_messages_pool["agent_response_technical"])
        if "{issue_type}" in agent_msg:
            agent_msg = agent_msg.format(
                issue_type=issue["problem_type"].replace("_", " ")
            )
        if "{date}" in agent_msg:
            agent_msg = agent_msg.format(date=issue["timestamp"].strftime("%Y-%m-%d"))

        messages.append(
            {
                "speaker": "agent",
                "timestamp": (timestamp_start + timedelta(minutes=2)).isoformat(),
                "message": agent_msg,
            }
        )
        # Weitere Konversation
        for i in range(random.randint(2, 5)):
            if i % 2 == 0:
                msg = random.choice(
                    chat_messages_pool["customer_polite"]
                    if not is_churned
                    else chat_messages_pool["customer_frustrated_technical"]
                )
                speaker = "customer"
            else:
                msg = random.choice(chat_messages_pool["agent_response_general"])
                speaker = "agent"

            messages.append(
                {
                    "speaker": speaker,
                    "timestamp": (
                        timestamp_start + timedelta(minutes=2 * (i + 2))
                    ).isoformat(),
                    "message": msg,
                }
            )

        duration_minutes = len(messages) * 2
        timestamp_end = timestamp_start + timedelta(minutes=duration_minutes)
        chat_transcripts.append(
            {
                "session_id": f"CHAT_{session_id:05d}",
                "customer_id": customer_id,
                "timestamp_start": timestamp_start,
                "timestamp_end": timestamp_end,
                "messages": json.dumps(messages),
                "agent_id": f"AGENT_{random.randint(1, 20):03d}",
                "resolution_status": random.choice(
                    ["resolved", "escalated", "unresolved"]
                )
            }
        )

        session_id += 1

        # Andere Chats (billing, allgemeine Fragen)
        for _ in range(n_other_chats):
            timestamp_start = datetime(2026, 3, 1) - timedelta(days=random.randint(1, 180))

            messages = []
            n_exchanges = random.randint(3, 6)

            for i in range(n_exchanges):
                if i % 2 == 0:
                    tone = random.choice(["customer_neutral", "customer_polite"])
                    message_text = random.choice(chat_messages_pool[tone])
                    speaker = "customer"
                else:
                    message_text = random.choice(
                        chat_messages_pool["agent_response_general"]
                    )
                    speaker = "agent"

            messages.append(
                {
                    "speaker": speaker,
                    "timestamp": (
                        timestamp_start + timedelta(minutes=i * 2)
                    ).isoformat(),
                    "message": message_text,
                }
            )

        duration_minutes = len(messages) * 2
        timestamp_end = timestamp_start + timedelta(minutes=duration_minutes)

        chat_transcripts.append(
            {
                "session_id": f"CHAT_{session_id:05d}",
                "customer_id": customer_id,
                "timestamp_start": timestamp_start,
                "timestamp_end": timestamp_end,
                "messages": json.dumps(messages),
                "agent_id": f"AGENT_{random.randint(1, 20):03d}",
                "resolution_status": random.choice(["resolved", "escalated"])
            }
        )

        session_id += 1
    # Andere Chats (billing, allgemeine Fragen)
    for _ in range(n_other_chats):
        timestamp_start = datetime(2026, 3, 1) - timedelta(days=random.randint(1, 180))

        messages = []
        n_exchanges = random.randint(3, 6)

        for i in range(n_exchanges):
            if i % 2 == 0:
                tone = random.choice(["customer_neutral", "customer_polite"])
                message_text = random.choice(chat_messages_pool[tone])
                speaker = "customer"
            else:
                message_text = random.choice(
                    chat_messages_pool["agent_response_general"]
                )
                speaker = "agent"

            messages.append(
                {
                    "speaker": speaker,
                    "timestamp": (
                        timestamp_start + timedelta(minutes=i * 2)
                    ).isoformat(),
                    "message": message_text,
                }
            )

        duration_minutes = len(messages) * 2
        timestamp_end = timestamp_start + timedelta(minutes=duration_minutes)

        chat_transcripts.append(
            {
                "session_id": f"CHAT_{session_id:05d}",
                "customer_id": customer_id,
                "timestamp_start": timestamp_start,
                "timestamp_end": timestamp_end,
                "messages": json.dumps(messages),
                "agent_id": f"AGENT_{random.randint(1, 20):03d}",
                "resolution_status": random.choice(["resolved", "escalated"])
            }
        )

        session_id += 1

chat_transcripts_df = pd.DataFrame(chat_transcripts)

print(f"\nChat Transcripts erstellt: {len(chat_transcripts_df)} Chats")
print(
    f"   Davon wegen technischen Problemen: {chat_transcripts_df['chat_reason'].value_counts().get('technical_issue', 0)}"
)

# Generate Agents

In [0]:
n_agents = 20

first_names = ['Max', 'Anna', 'Lukas', 'Sophie', 'Tim', 'Laura', 'Felix', 'Emma', 
               'Paul', 'Mia', 'Leon', 'Hannah', 'Jonas', 'Lena', 'David', 
               'Sarah', 'Ben', 'Julia', 'Noah', 'Lisa']

last_names = ['Müller', 'Schmidt', 'Schneider', 'Fischer', 'Weber', 'Meyer', 
              'Wagner', 'Becker', 'Schulz', 'Hoffmann', 'Koch', 'Bauer',
              'Richter', 'Klein', 'Wolf', 'Schröder', 'Neumann', 'Schwarz',
              'Zimmermann', 'Braun']

agent_names = [f"{first_names[i]} {last_names[i]}" for i in range(n_agents)]

agent_profile = pd.DataFrame({
    'agent_id': [f'AGENT_{i:03d}' for i in range(1, n_agents + 1)],
    'agent_name': agent_names,
    'employment_date': pd.date_range(end='2026-03-01', periods=n_agents, freq='45D'), 
    'experience_level': np.random.choice(['Junior', 'Mid', 'Senior', 'Lead'], 
                                        n_agents, p=[0.3, 0.45, 0.2, 0.05])
})

# Berechne Beschäftigungsdauer in Monaten
agent_profile['employment_months'] = (
    (pd.Timestamp('2026-03-01') - agent_profile['employment_date']).dt.days / 30
).round(1)

# Gehalt basierend auf Experience Level und Department
salary_base = {
    'Junior': 2800,
    'Mid': 3500,
    'Senior': 4500,
    'Lead': 5500
}

agent_profile['monthly_salary_eur'] = agent_profile.apply(
    lambda row: salary_base[row['experience_level']] + 
                np.random.randint(-200, 300),
    axis=1
)

print("Agent Profile erstellt")

# Save Data

In [0]:
customer_profile.to_csv('customer_profile.csv', index=False)
churn_labels.to_csv('churn_labels.csv', index=False)
#support_tickets.to_csv('support_tickets.csv', index=False)
#chat_transcripts_df.to_csv('chat_transcripts.csv', index=False)
connection_quality_logs.to_csv('connection_quality_logs.csv', index=False)
agent_profile.to_csv('agent_profile.csv', index=False)

print(f"\nZusammenfassung:")
print(f"   • Customers: {len(customer_profile)}")
print(f"   • Churned Customers: {n_churned} ({churn_rate*100}%)")
#print(f"   • Support Tickets: {len(support_tickets)}")
#print(f"   • Chat Transcripts: {len(chat_transcripts_df)}")
print(f"   • Connection Quality Logs: {len(connection_quality_logs)}")
print(f"     - Mit erkannten Problemen: {(connection_quality_logs['issue_detected'] != 'none').sum()}")